# 🧬 Translation Efficiency Prediction with OmniGenBench

Welcome! This tutorial will guide you through the process of predicting **Translation Efficiency (TE)** from mRNA sequences using **OmniGenBench**. We'll cover the entire workflow, from understanding the biological problem to using a fine-tuned Genomic Foundation Model (GFM) to make predictions.

### 1. The Biological Challenge: What is Translation Efficiency?

Translation is the fundamental process where a cell's ribosome reads an mRNA sequence and synthesizes a protein. **Translation Efficiency (TE)** measures how effectively this process occurs for a given mRNA molecule. High TE means more protein is produced, while low TE means less.

Understanding and predicting TE is crucial for:
- **Synthetic Biology**: Designing genes that produce a desired amount of protein.
- **Disease Research**: Understanding how mutations can affect protein levels and lead to disease.
- **Agriculture**: Engineering crops with enhanced traits.

Predicting TE from an mRNA sequence alone is challenging because it depends on complex features within the sequence. This is an ideal problem for deep learning to solve.

### 2. The Data: Rice Translation Efficiency Dataset

We will use a dataset containing mRNA sequences from *Oryza sativa* (rice).
- **What it contains**: Each entry is an mRNA sequence.
- **What it labels**: Each sequence is labeled as either "High TE" (1) or "Low TE" (0).
- **Our Goal**: To train a model that can take any rice mRNA sequence and classify it as having high or low translation efficiency.

### 3. The Tool: From Language Models to Genomic Foundation Models (GFMs)

The same principles that power Natural Language Processing (NLP) models like BERT can be applied to genomics. The "language of life" is written in the sequence of nucleotides (A, C, G, T/U). **Genomic Foundation Models (GFMs)**, like **OmniGenome**, are large-scale models pre-trained on massive amounts of genomic data.

By learning the "grammar" of DNA and RNA, these models develop a powerful, generalized understanding of genomics. This allows them to be fine-tuned for specific tasks like TE prediction with high accuracy and less data than training from scratch.

### 4. The Workflow: A 4-Step Guide to Fine-Tuning

We will follow a standard 4-step fine-tuning pipeline, which is a common practice in machine learning.

```mermaid
flowchart TD
    subgraph "4-Step Workflow for TE Prediction"
        A["📥 Step 1: Data Preparation<br/>Download and process the TE dataset"] --> B["🔧 Step 2: Model Initialization<br/>Load the pre-trained OmniGenome model"]
        B --> C["🎓 Step 3: Model Training<br/>Fine-tune the model on the TE dataset"]
        C --> D["🔮 Step 4: Model Inference<br/>Use the trained model to predict TE on new sequences"]
    end

    style A fill:#e1f5fe,stroke:#333,stroke-width:2px
    style B fill:#f3e5f5,stroke:#333,stroke-width:2px
    style C fill:#e8f5e8,stroke:#333,stroke-width:2px
    style D fill:#fff3e0,stroke:#333,stroke-width:2px
```

Let's get started!

## 🚀 Step 1: Data Preparation

First, we need to set up our environment and prepare the data.

### 1.1: Environment Setup
Install and import the necessary libraries. `omnigenbench` is our core tool.

In [ ]:
!pip install torch transformers pandas autocuda omnigenbench findfile requests -U

In [ ]:
import os
import torch
from omnigenbench import OmniGenomeForSequenceClassification, OmniTokenizer
from omnigenbench import ClassificationMetric, Trainer, ModelHub
from omnigenbench import OmniDatasetForSequenceClassification
import warnings
from autocuda import auto_cuda

warnings.filterwarnings('ignore')

### 1.2: Data Preparation with OmniDataset

With the enhanced `omnigenbench` framework, data preparation is now much simpler! The base `OmniDataset` class now includes:

- **`from_huggingface()` Class Method**: Automatically downloads datasets from HuggingFace Hub
- **Automatic Download**: Downloads and extracts datasets if not found locally  
- **Built-in DataLoader**: Provides `.get_dataloader()` method with optimized settings
- **Universal Interface**: Works with any OmniDataset subclass (SequenceClassification, Regression, etc.)

This eliminates the need for custom dataset classes and manual download functions!

## 🚀 Step 2, 3 & 4: Model Training and Inference

Now we'll combine the remaining steps into a single, powerful workflow. We will define a configuration, then initialize the model, train it, and finally use it for inference.

### 2.1: Global Configuration
We centralize all parameters for reproducibility.

In [ ]:
# Model and Tokenizer
model_name_or_path = "yangheng/OmniGenome-52M"

# Data - OmniDataset will handle download automatically
cache_dir = "TE_data" # default dir path

# Training parameters
num_labels = 2
epochs = 3
batch_size = 16
learning_rate = 1e-5
patience = 3 # default

# Output
save_dir = "TE_model_complete" # default dir path 

os.makedirs(save_dir, exist_ok=True)

print("⚙️ Configuration set - OmniDataset will handle data download automatically!")

### 2.2 & 3.1: Simplified Model Training Pipeline

Thanks to the enhanced `OmniDataset` base class, the training pipeline is now much cleaner! Here's what happens:

1. **Universal Interface**: `OmniDatasetForSequenceClassification.from_huggingface()` works for any dataset
2. **Automatic Download**: Downloads and processes the dataset automatically if not found locally
3. **Built-in DataLoaders**: `.get_dataloader()` method creates optimized DataLoaders
4. **Generic Solution**: Works with any genomic task (classification, regression, token-level, etc.)

Let's see this universal approach in action:

In [ ]:
# 1. Initialize Tokenizer 
tokenizer = OmniTokenizer.from_pretrained(model_name_or_path)

# 2. Create datasets using the enhanced OmniDataset.from_huggingface interface
print("🏗️ Creating datasets with automatic download...")
datasets = OmniDatasetForSequenceClassification.from_huggingface(
    dataset_name="yangheng/translation_efficiency_prediction",
    tokenizer=tokenizer,
    max_length=512,
    cache_dir=cache_dir
) # can we also use local path?

# 3. Get DataLoaders directly from datasets
train_loader = datasets['train'].get_dataloader(batch_size=batch_size, shuffle=True)
valid_loader = datasets['valid'].get_dataloader(batch_size=batch_size, shuffle=False)
test_loader = datasets['test'].get_dataloader(batch_size=batch_size, shuffle=False)

print(f"📊 Dataset sizes:")
print(f"  📈 Training samples: {len(datasets['train'])}")
print(f"  🔍 Validation samples: {len(datasets['valid'])}")
print(f"  🧪 Test samples: {len(datasets['test'])}")

# 4. Initialize Model
model = OmniGenomeForSequenceClassification.from_pretrained(
    model_name_or_path, 
    num_labels=num_labels
)

# 5. Initialize Optimizer and Metrics
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
compute_metrics = [ClassificationMetric(num_classes=num_labels)]

# 6. Initialize and run Trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    eval_loader=valid_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    epochs=epochs,
    patience=patience,
    compute_metrics=compute_metrics,
)

print("Starting model training...")
training_results = trainer.train(path_to_save=save_dir)
print("Training finished.")
print("Best validation metrics:", trainer.best_metric)
print("Final test metrics:", trainer.metrics['test'])

# 7. Save the final model
hub = ModelHub()
hub.save(model=trainer.model, save_dir=save_dir)
print(f"Model successfully saved to {save_dir}")

### 4.1: Model Inference
Finally, let's use our fine-tuned model to make predictions on new sequences. We'll use the `ModelHub` one-liner for simplicity.

In [ ]:
# Load the fine-tuned model from the local directory
inference_model = ModelHub.load(save_dir).to(auto_cuda())
print("Inference model loaded successfully.")

# Define two example sequences
high_te_sequence = "AAACCAACAAAATGCAGTAGAAGTACTCTCGAGCTATAGTCGCGACGTGCTGCCCCGCAGGAGTACAGTAGTAGTACAACGTAAGCGGGAGCAACAGACTCCCCCCCTGCAACCCACTGTGCCTGTGCCCTCGACGCGTCTCCGTCGCTTTGGCAAATGTCACGTACATATTACCGTCTCAGGCTCTCAGCCATGCTCCCTACCACCCCTGCAGCGAAGCAAAAGCCACGCACGCGGCGCCTGACATGTAACAGGACTAGACCATCTTGTTCATTTCCCGCACCCCCTCCTCTCCTCTTCCTCCATCTGCCTCTTTAAAACAGTAAAAATAACCGTGCATCCCCTGGGCAAAATCTCTCCCATACATACACTACAGCGGCGAACCTTTCCTTATTCTCGCAACGCCTCGGTAACGGGCAGCGCCTGCTCCGCGCCGCGGTTGCGAGTTCGGGAAGGCGGCCGGAGTCGCGGGGAGGAGAGGGAGGATTCGATCGGCCAGA"
low_te_sequence = "TGGAGATGGGCAGATGGCACACAAAACATGAATAGAAAACCCAAAAGGAAGGATGAAAAAAACACACACACACACACACACAAAACACAGAGAGAGAGAGAGAGAGAGCGAGAAAAGAAAAGAAAAAACCAATTCTTTTGGTCTCTTCCCTCTCCGTTTGTCGTGTCGAAGCCTTTGCCCCCACCACCTCCTCCTCTCCTCTCCCTTCCTCCCCTCCTCCCCATCTCGCTCTCCTCCCTCCTCTCTCCTCTCCTCGTCTCCTCTTCCTCTCCATTCCATTGGCCATTCCATTCCATTCCACCCCCCATGAAACCCCAAACCCTCGTCGGCCTCGCCGCGCTCGCGTAGCGCACCCGCCCTTCTCCTCTCGCCGGTGGTCCGCCGCCAGCCTCCCCCCACCCGATCCCGCCGCCCCCCCCGCCTTCACCCCGCCCACGCGGACGCATCCGATCCCGCCGCATCGCCGCGCGGGGGGGGGGGGGGGGGGGGGGGGGAGGGCACG"

# Run inference
high_te_output = inference_model.inference(high_te_sequence)
low_te_output = inference_model.inference(low_te_sequence)

print(f"Prediction for High-TE sequence: {high_te_output['predictions']} (Predicted Label: '{high_te_output['label']}')")
print(f"Prediction for Low-TE sequence: {low_te_output['predictions']} (Predicted Label: '{low_te_output['label']}')")

## 🎉 Tutorial Summary and Next Steps

Congratulations! You have successfully completed this tutorial using the enhanced OmniGenBench framework with universal dataset support.

### What You've Learned:

1. **Universal Dataset Interface**: Used `OmniDataset.from_huggingface()` for automatic dataset handling
2. **Generic Solution**: The same interface works for all genomic tasks (classification, regression, token-level)
3. **Streamlined Workflow**: Experienced a much cleaner and more efficient training pipeline
4. **Foundation Model Training**: Successfully fine-tuned OmniGenome for TE prediction

### Key Improvements in This Version:

- ✅ **Universal Interface**: One `from_huggingface()` method works for all dataset types
- ✅ **No Specialized Classes**: No need for task-specific dataset classes
- ✅ **Automatic Downloads**: Built into the base OmniDataset class
- ✅ **Cleaner Architecture**: More maintainable and extensible design
- ✅ **Flexible Usage**: Easy to adapt for any HuggingFace genomic dataset
le and consistent across all tasks! 🧬✨